# Data Migration: SQL Server to Postgres

In [1]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

In [2]:
os.getcwd()

'c:\\Users\\opeab\\OneDrive\\Documents\\Github\\sql-server-to-postgres-migration'

In [3]:
os.listdir()

['.env',
 '.git',
 '.gitignore',
 '.venv',
 'generate_data.py',
 'README.md',
 'run_migration_uat.ipynb']

In [4]:
os.path.isfile(".env")

True

## 1. Load credentials

In [5]:
load_dotenv(".env")

True

In [6]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_db = os.getenv("SQL_SERVER_DB")

In [7]:
print(f"SQL SERVER HOST: {sql_host}")
print(f"SQL SERVER DB: {sql_db}")

SQL SERVER HOST: OPSY\SQLEXPRESS
SQL SERVER DB: TransactionDB_UAT


In [8]:
pg_host = os.getenv("POSTGRES_HOST")
pg_port = os.getenv("POSTGRES_PORT")
pg_db = os.getenv("POSTGRES_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")

In [9]:
print(f"POSTGRES HOST: {pg_host}")
print(f"POSTGRES PORT: {pg_port}")
print(f"POSTGRES DB: {pg_db}")
print(f"POSTGRES USER: {pg_user}")
print(f"POSTGRES PASSWORD: {pg_password}")

POSTGRES HOST: localhost
POSTGRES PORT: 5432
POSTGRES DB: transaction_uat
POSTGRES USER: postgres
POSTGRES PASSWORD: Positive*1


## 2. Connect to SQL Server

In [10]:
print("Connecting  to SQL Server...")
print(f"   Server: {sql_host}")
print(f"   Database: {sql_db}")

Connecting  to SQL Server...
   Server: OPSY\SQLEXPRESS
   Database: TransactionDB_UAT


In [11]:
try:
    sql_conn_string = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_db};"
        "Trusted_Connection=yes;"
    )

    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print("[SUCCESS] SQL Server connection established.")

except Exception as e:
    print(f"SQL Server connection failed: {e}")
    print(""" How to troubleshoot:
          > 1. Check server name in .env file is correct
          . 2. Verify SQL Server is running
          > 3. Check Windows Authentication is enabled
            ....
""")

[SUCCESS] SQL Server connection established.


# 3. Connect to PostgreSQL

In [12]:
print("Connecting to PostgreSQL...")
print(f"    Server: {pg_host}")
print(f"    Database: {pg_db}")

Connecting to PostgreSQL...
    Server: localhost
    Database: transaction_uat


In [13]:
try: 
    pg_conn = psycopg2.connect(
        host=pg_host,
        port=pg_port,
        database=pg_db,
        user=pg_user,
        password=pg_password
    )

    pg_cursor=pg_conn.cursor()
    pg_cursor.execute("SELECT version();")

    pg_version = pg_cursor.fetchone()[0]

    print("Connected to PostgreSQL successfully!")
    print(f"    Version: {pg_version[:50]}...\n")


except psycopg2.OperationalError as e:
    print(f"Postgres connection failed:{e}")
    print(""" How to troubleshoot:
          > 1. Check Postgres is running
          > 2. Verify username + password
          > 3. Check database exists
        ....

""")
    
except Exception as e:
    print(f" Unexpected error: {e}")
    raise

Connected to PostgreSQL successfully!
    Version: PostgreSQL 18.1 on x86_64-windows, compiled by msv...



# 4. Define the tables to migrate

### Migration order

- Categories (no dependencies)
- Supplies (no dependencies)
- Customers (no dependencies)
- Products (depends on Categories and Suppliers)


In [14]:
tables_to_migrate = ['Categories', 'Suppliers', 'Customers', 'Products']
print(tables_to_migrate)

['Categories', 'Suppliers', 'Customers', 'Products']


In [15]:
print("Table to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}. {table}")

total_no_tbls = len(tables_to_migrate)
print(f"\nTotal no of tables to migrate: {total_no_tbls}")

Table to migrate:
    1. Categories
    2. Suppliers
    3. Customers
    4. Products

Total no of tables to migrate: 4


# 5. Run pre-migration checks

In [16]:
print("=" * 50)
print(">>> Check 1: ROW COUNTS")
print("=" * 50)

>>> Check 1: ROW COUNTS


In [17]:
baseline_counts = {}


try:
    for table in tables_to_migrate:
        quoted_table = f"[{table}]"
        row_count_query = f"SELECT COUNT(*) as total_rows FROM {quoted_table};"
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:15} {count:>12} rows")

    total_rows = sum(baseline_counts.values())
    print(f"{'-' * 30}")
    print(f"{'TOTAL':15} {total_rows:>12} rows")
    print("\n Baseline captured! ")

except Exception as e:
    print(f"Failed to get baseline counts: {e}")
    raise

Categories                 8 rows
Suppliers               5000 rows
Customers             900000 rows
Products              150000 rows
------------------------------
TOTAL                1055008 rows

 Baseline captured! 


In [18]:
tables_to_migrate = {'Categories', 'Suppliers', 'Customers', 'Products'}
print(tables_to_migrate)

{'Products', 'Suppliers', 'Customers', 'Categories'}


In [19]:
print("Table to migrate:")
for i, table in enumerate(tables_to_migrate, 1):
    print(f"    {i}.  {table}")

print(f"\nTotal no of tables to migrate: {len(tables_to_migrate)}")

Table to migrate:
    1.  Products
    2.  Suppliers
    3.  Customers
    4.  Categories

Total no of tables to migrate: 4


# 5. Run pre-migration checks

In [20]:
print("=" * 50)
print(">>> ROW COUNTS")
print("=" * 50)

>>> ROW COUNTS


In [21]:
test_query = "SELECT COUNT(*) AS total_rows FROM Categories;"
sql_cursor.execute(test_query)

count = sql_cursor.fetchone()[0]

print(f"Results: {count}")

Results: 8


In [22]:
baseline_counts ={}


try:
    for table in tables_to_migrate:
        row_count_query = f"SELECT COUNT(*) AS total_rows FROM {table}"#(Warning: Do not input SQL queries with f-strings in production. Malicious input can lead to SQL injection attacks. Always use parameterized queries or proper sanitization.)
        sql_cursor.execute(row_count_query)
        count = sql_cursor.fetchone()[0]

        baseline_counts[table] = count
        print(f"{table:15} {count:>12} rows")
        
        
    baseline_counts[table] = count
    print(f"{'-' * 30}")
    print(f"{'TOTAL':15} {total_rows:>12} rows")
    print("\n Baseline captured! ")

except Exception as e:
    print("Failed to get baseline counts: {e}")
    raise

Products              150000 rows
Suppliers               5000 rows
Customers             900000 rows
Categories                 8 rows
------------------------------
TOTAL                1055008 rows

 Baseline captured! 


In [23]:
print("=" * 50)
print(">>> Check 2: NULL COUNTS (CustomerName)")
print("=" * 50)

quality_issues = []

>>> Check 2: NULL COUNTS (CustomerName)


In [ ]:
try:
    print("\nCheck 2: NULL COUNTS (CustomerName)")
    sql_cursor.execute("""SELECT COUNT(*) AS null_count
                            FROM Customers
                            WHERE CustomerName IS NULL""")
    null_names = sql_cursor.fetchone()[0]
    if null_names > 0:
        quality_issues.append(f"    > {null_names:,} customers with NULL names...")
    # print(quality_issues)

    print("\nCHECK 3: Invalid email format check")
    sql_cursor.execute("""SELECT COUNT(*) AS invalid_email_count
                            FROM Customers
                            WHERE Email LIKE '%@invalid'   """)
    invalid_emails = sql_cursor.fetchone()[0]
    if invalid_emails > 0:
        quality_issues.append(f"    > {invalid_emails:,} email with invalid email formats...")
    # print(quality_issues)

    print("\nCHECK 4: NEGATIVE PRODUCT PRICES")
    sql_cursor.execute("""SELECT COUNT(*) AS negative_price_count
                            FROM Products
                            WHERE UnitPrice < 0""")
    negative_price = sql_cursor.fetchone()[0]
    if negative_price > 0:
        quality_issues.append(f"    > {negative_price:,} prices contain negative values...")
    # print(quality_issues)

    print("\nCHECK 5: NEGATIVE STOCK QUANTITIES")
    sql_cursor.execute("""SELECT COUNT(*) AS negative_stock_quantities_count
                            FROM Products
                            WHERE StockQuantity < 0
                        """ )
    negative_stock_quantities = sql_cursor.fetchone()[0]
    if negative_stock_quantities > 0:
        quality_issues.append(f"    > {negative_stock_quantities:,} products with negative stock...")
    #print(quality_issues)

    print("\nCHECK 6: ORPHANED FOREIGN KEYS")
    sql_cursor.execute("""SELECT COUNT(*) AS orphaned_records
                            FROM Products prod
                            WHERE NOT EXISTS (SELECT 1
                                                FROM Suppliers sup
                                                WHERE sup.SupplierID = prod.SupplierID)
                       """)
    orphaned_fks = sql_cursor.fetchone()[0]
    if orphaned_fks > 0:
        quality_issues.append(f"    > {orphaned_fks:,} products with orphaned foreign keys...")
    # print(quality_issues)


    print("\nCHECK 7: FUTURE DATES CHECK")
    sql_cursor.execute("""SELECT COUNT(*) AS future_dates_count
                            FROM Customers
                            WHERE CreatedDate > GETDATE()
                       """)
    future_dates = sql_cursor.fetchone()[0]
    if future_dates > 0:
        quality_issues.append(f"   > {future_dates:,} customer with future creations data later than current date...")
    # print(quality_issues)

    if quality_issues:
        print("\nData quality issues found (will migrate as-is)")
        for issue in quality_issues:
            print(issue)
    else: 
        print("No data quality issues identified!")


except Exception as e:
    print(f"[ERROR] ===> Unexpected issue: {e}")
    raise


Check 2: NULL COUNTS (CustomerName)

CHECK 3: Invalid email format

CHECK 4: NEGATIVE PRODUCT PRICES

CHECK 5: NEGATIVE STOCK QUANTITIES

CHECK 6: ORPHANED FOREIGN KEYS

CHECK 7: FUTURE DATES CHECK

Data quality issues found (will migrate as-is)
    - 4,514 customers with NULL names...
    - 8,844 email with invalid email formats...
    - 775 prices contain negative values...
    1,467 products with negative stock...
    24,700 products with orphaned foreign keys...
    7,333 customer with future creations data later than current date...
    140,471 customers with duplicate names...
    - 4,514 customers with NULL names...
    - 8,844 email with invalid email formats...
    - 775 prices contain negative values...
    1,467 products with negative stock...
    24,700 products with orphaned foreign keys...
    7,326 customer with future creations data later than current date...
    - 4,514 customers with NULL names...
    - 8,844 email with invalid email formats...
    - 775 prices conta

In [ ]:
    print("\nCHECK 8: DUPLICATE CUSTOMER NAMES")
    sql_cursor.execute("""SELECT COUNT(*) AS duplicate_customer_count
                            FROM (SELECT CustomerName, COUNT(*) AS count
                                    FROM Customers
                                    GROUP BY CustomerName
                                    HAVING COUNT(*) > 1) AS duplicates""")
    duplicate_customers = sql_cursor.fetchone()[0]
    if duplicate_customers > 0:
        quality_issues.append(f"    {duplicate_customers:,} customers with duplicate names...")
   print(quality_issues)